# Importe des Variables

In [ ]:
import pickle
from pathlib import Path

data_dir = Path("Data")
data_dir.mkdir(exist_ok=True)
with open(data_dir / "ffor_data.pkl", "rb") as f_pkl:
    d = pickle.load(f_pkl)

# Remettre toutes les variables dans le namespace local
locals().update(d)

scenario_file = data_dir / "ffor_grid_scenarios.pkl"
if not scenario_file.exists():
    raise FileNotFoundError(f"Impossible de trouver {scenario_file}. Execute d'abord grid_generation_belalp.ipynb.")
with open(scenario_file, "rb") as f_pkl:
    scenarios = pickle.load(f_pkl)

# FFOR temporel local: reseau MV sans Belalp. Pour tester l'autre topologie,
# utiliser "with_belalp"; la disponibilite Belalp reste alors la moyenne statique.
scenario_key = "without_belalp"
scenario = scenarios[scenario_key]
nodes = scenario["nodes"]
n_nodes = scenario["n_nodes"]
pcc_bus = scenario["pcc_bus"]
P_ref = scenario["P_ref"]
Q_ref = scenario["Q_ref"]
J_Ptheta = scenario["J_Ptheta"]
J_PU = scenario["J_PU"]
J_Qtheta = scenario["J_Qtheta"]
J_QU = scenario["J_QU"]
line_data = scenario["line_data"]
P_load = scenario["P_load"]
Q_load = scenario["Q_load"]
S_inv_max = scenario["S_inv_max"]
V = scenario["V"]
delta_Umin = scenario["delta_Umin"]
delta_Umax = scenario["delta_Umax"]
P_base = scenario["P_base"]
Q_base = scenario["Q_base"]

print(f"✅ Données chargées: {scenario['name']} | {n_nodes} bus MV | PCC bus {pcc_bus}")

# 1. FFOR pour dates/heures clefs

| Saison | Mois |
|--------|------|
| ☀️ Été | Juin, Juillet, Août |
| ❄️ Hiver | Décembre, Janvier, Février |

**Heures étudiées** : 07h00 · 12h00 · 18h00 · 00h00

## Créer les filtres temporels


**CAS où P_load varie avec le temps**

In [ ]:
import numpy as np
import pandas as pd

# Parametre explicite: "fixed" ou "temporal".
# Il remplace la selection implicite qui dependait auparavant de l'ordre
# d'execution des deux cellules de snapshots.
FFOR_LOAD_MODE = "fixed"

ETE_MONTHS = [6, 7, 8]
HIVER_MONTHS = [12, 1, 2]
HEURES = [7, 12, 18, 0]
SAISONS = {"Été": ETE_MONTHS, "Hiver": HIVER_MONTHS}

times = pd.date_range("2023-01-01 00:00", periods=8760, freq="h")
npro_months = times.month
npro_hours = times.hour

P_pv_dt_df = P_pv_max_dt.reset_index().pivot_table(
    index="time", columns="bus", values="P_pv"
)
P_hp_dt_df = P_hp_max_dt.reset_index().pivot_table(
    index="time", columns="bus", values="P_hp"
)

alpha_min = 0.85
alpha_max = 1.20
f_min = float(np.min(f_load_elec))
f_max = float(np.max(f_load_elec))


def reduce_profile(values, method):
    if method == "mean":
        return values.mean()
    if method == "min":
        return values.min()
    raise ValueError(f"Methode de reduction inconnue: {method}")


def build_pv_available_snapshot_for_scenario(
    scenario_selected,
    t_indices,
    method="mean",
):
    scenario_nodes = list(map(int, scenario_selected["nodes"]))
    selected = P_pv_dt_df.iloc[t_indices]
    reduced = reduce_profile(selected, method)
    values = reduced.reindex(scenario_nodes, fill_value=0.0).to_dict()

    # Les PV locaux disposent deja d'un profil horaire. BelalpSolar est un bus
    # ajoute apres la creation de ffor_data.pkl: son profil est reconstruit avec
    # l'irradiance normalisee du bus geographique de raccordement le plus proche.
    metadata = scenario_selected.get("metadata", {})
    belalp_bus = metadata.get("belalp_bus")
    nearest_bus = metadata.get("nearest_bus")
    if belalp_bus is not None and nearest_bus is not None:
        source_nodes = list(map(int, d["nodes"]))
        nearest_position = source_nodes.index(int(nearest_bus))
        irradiance = np.asarray(d["G_norm_node_dt"])[t_indices, nearest_position]
        irradiance_factor = float(reduce_profile(irradiance, method))
        values[int(belalp_bus)] = (
            float(metadata["belalp_p_mwp"]) * irradiance_factor
        )

    for bus in scenario_nodes:
        if bus not in P_pv_dt_df.columns and bus != belalp_bus:
            values[bus] = float(
                scenario_selected["P_pv_available"].get(bus, 0.0)
            )
    return values


def load_alpha_for_indices(t_indices):
    f_snapshot = float(np.mean(np.asarray(f_load_elec)[t_indices]))
    if np.isclose(f_max, f_min):
        return 1.0, f_snapshot
    alpha_snapshot = alpha_min + (alpha_max - alpha_min) * (
        f_snapshot - f_min
    ) / (f_max - f_min)
    return float(alpha_snapshot), f_snapshot


def build_snapshots_for_scenario(scenario_selected, load_mode):
    if load_mode not in {"fixed", "temporal"}:
        raise ValueError("load_mode doit valoir 'fixed' ou 'temporal'.")

    scenario_nodes = list(map(int, scenario_selected["nodes"]))
    scenario_snapshots = {}
    for saison, months in SAISONS.items():
        for heure in HEURES:
            mask = npro_months.isin(months) & (npro_hours == heure)
            t_indices = np.where(mask)[0]

            p_pv_available = build_pv_available_snapshot_for_scenario(
                scenario_selected,
                t_indices,
                method="mean",
            )
            p_hp_min = (
                P_hp_dt_df.iloc[t_indices]
                .mean()
                .reindex(scenario_nodes, fill_value=0.0)
                .to_dict()
            )
            q_pv_max = {
                bus: float(scenario_selected["Q_pv_max"].get(bus, 0.0))
                for bus in scenario_nodes
            }

            if load_mode == "temporal":
                alpha_snapshot, f_snapshot = load_alpha_for_indices(t_indices)
            else:
                alpha_snapshot, f_snapshot = 1.0, np.nan

            scenario_snapshots[(saison, heure)] = {
                "P_pv_available": p_pv_available,
                "Q_pv_max": q_pv_max,
                "P_hp_max": p_hp_min,
                "P_load": {
                    bus: float(scenario_selected["P_load"].get(bus, 0.0))
                    * alpha_snapshot
                    for bus in scenario_nodes
                },
                "Q_load": {
                    bus: float(scenario_selected["Q_load"].get(bus, 0.0))
                    * alpha_snapshot
                    for bus in scenario_nodes
                },
                "alpha": alpha_snapshot,
                "f_snap": f_snapshot,
                "load_mode": load_mode,
            }
    return scenario_snapshots


def snapshots_to_frame(scenario_selected, scenario_snapshots):
    rows = []
    for (saison, heure), snapshot in scenario_snapshots.items():
        for bus in scenario_selected["nodes"]:
            rows.append({
                "saison": saison,
                "heure": heure,
                "bus": bus,
                "load_mode": snapshot["load_mode"],
                "alpha": snapshot["alpha"],
                "P_pv_available": snapshot["P_pv_available"].get(bus, 0.0),
                "Q_pv_max": snapshot["Q_pv_max"].get(bus, 0.0),
                "S_inv_max": scenario_selected["S_inv_max"].get(bus, 0.0),
                "P_hp_max": snapshot["P_hp_max"].get(bus, 0.0),
                "P_load": snapshot["P_load"].get(bus, 0.0),
                "Q_load": snapshot["Q_load"].get(bus, 0.0),
            })
    return pd.DataFrame(rows)


snapshots_variable_load = build_snapshots_for_scenario(
    scenario,
    load_mode="temporal",
)
display(
    snapshots_to_frame(scenario, snapshots_variable_load)
    .groupby(["saison", "heure"])[
        ["P_pv_available", "P_hp_max", "P_load", "alpha"]
    ]
    .mean()
)


**Cas où P_load ne varie pas avec le temps**

In [ ]:
snapshots_fixed_load = build_snapshots_for_scenario(
    scenario,
    load_mode="fixed",
)

if FFOR_LOAD_MODE == "temporal":
    snapshots = snapshots_variable_load
elif FFOR_LOAD_MODE == "fixed":
    snapshots = snapshots_fixed_load
else:
    raise ValueError("FFOR_LOAD_MODE doit valoir 'fixed' ou 'temporal'.")

df_snapshots = snapshots_to_frame(scenario, snapshots)
df_snapshots.to_csv(
    data_dir / f"snapshots_{FFOR_LOAD_MODE}.csv",
    index=False,
)

display(
    df_snapshots.groupby(["saison", "heure"])[
        ["P_pv_available", "P_hp_max", "P_load", "alpha"]
    ].mean()
)
print(
    f"{len(snapshots)} snapshots selectionnes | "
    f"mode de charge: {FFOR_LOAD_MODE} | "
    f"fichier: Data/snapshots_{FFOR_LOAD_MODE}.csv"
)


## FFOR adapte et sorties CSV

Le solveur reprend la formulation statique : reseau MV vu depuis le bus 19, bilans nodaux, quatre Jacobiennes analytiques, limites de lignes sur le flux total, tensions, slack PCC et cercle de capacite de chaque onduleur PV.

`FFOR_LOAD_MODE` choisit explicitement entre charges fixes et charges temporelles. Les snapshots modifient les disponibilites PV et HP. Pour BelalpSolar, la puissance active temporelle suit l'irradiance normalisee du bus de raccordement le plus proche.


In [ ]:
import copy
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandapower as pp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

n_angles = 72
angles   = np.linspace(0, 2*np.pi, n_angles, endpoint=False)

# ── Fonction FFOR ─────────────────────────────────────────────────────────────
def solve_ffor_snapshot(a, b, snap):
    model = gp.Model("FFOR_temporal")
    model.Params.OutputFlag     = 0
    model.Params.DualReductions = 0

    bus_position = {bus: index for index, bus in enumerate(nodes)}
    Pk      = model.addVars(nodes, lb=-GRB.INFINITY, name="Pk")
    Qk      = model.addVars(nodes, lb=-GRB.INFINITY, name="Qk")
    theta   = model.addVars(nodes, lb=-GRB.INFINITY, name="theta")
    delta_U = model.addVars(nodes, lb=-GRB.INFINITY, name="U")
    P_pv    = model.addVars(nodes, lb=0,             name="Ppv")
    Q_pv    = model.addVars(nodes, lb=-GRB.INFINITY, name="Qpv")
    P_hp    = model.addVars(nodes, lb=-GRB.INFINITY, ub=0.0, name="Php")

    for k in nodes:
        if k == pcc_bus:
            continue
        model.addConstr(Pk[k] == P_pv[k] + P_hp[k] + snap["P_load"].get(k, 0))
        model.addConstr(Qk[k] == Q_pv[k] + snap["Q_load"].get(k, 0))

    for k in nodes:
        position = bus_position[k]
        model.addConstr(Pk[k] == P_ref[position] + gp.quicksum(J_Ptheta[position, other_position] * theta[j] + J_PU[position, other_position] * delta_U[j] for other_position, j in enumerate(nodes)))
        model.addConstr(Qk[k] == Q_ref[position] + gp.quicksum(J_Qtheta[position, other_position] * theta[j] + J_QU[position, other_position] * delta_U[j] for other_position, j in enumerate(nodes)))

    for k in nodes:
        model.addConstr(P_pv[k] >= 0.0)
        model.addConstr(P_pv[k] <= snap["P_pv_available"].get(k, 0))
        model.addConstr(Q_pv[k] <=  snap["Q_pv_max"].get(k, 0))
        model.addConstr(Q_pv[k] >= -snap["Q_pv_max"].get(k, 0))
        model.addConstr(P_hp[k] >= snap["P_hp_max"].get(k, 0))
        model.addConstr(
            P_hp[k] <= snap.get("P_hp_upper", {}).get(k, 0.0)
        )

    for _, row in line_data.iterrows():
        i, j = int(row["from_bus"]), int(row["to_bus"])
        angle_delta = theta[i] - theta[j]
        voltage_delta = delta_U[i] - delta_U[j]
        Pij = (
            float(row["P_base"])
            - float(row["b"]) * angle_delta
            + float(row["g"]) * voltage_delta
        )
        Qij = (
            float(row["Q_base"])
            - float(row["g"]) * angle_delta
            - float(row["b"]) * voltage_delta
        )
        model.addQConstr(Pij * Pij + Qij * Qij <= float(row["S_max"]) ** 2)

    for k in nodes:
        model.addConstr(V + delta_U[k] >= delta_Umin * V)
        model.addConstr(V + delta_U[k] <= delta_Umax * V)

    model.addConstr(theta[pcc_bus]   == 0)
    model.addConstr(delta_U[pcc_bus] == 0)

    for k in nodes:
        inverter_smax = float(S_inv_max.get(k, 0.0))
        if inverter_smax > 1e-9:
            model.addQConstr(P_pv[k]**2 + Q_pv[k]**2 <= inverter_smax**2)
        else:
            model.addConstr(P_pv[k] == 0.0)
            model.addConstr(Q_pv[k] == 0.0)

    model.setObjective(a * Pk[pcc_bus] + b * Qk[pcc_bus], GRB.MINIMIZE)
    model.optimize()

    if model.status == GRB.OPTIMAL:
        return {
            "P_linear": Pk[pcc_bus].X,
            "Q_linear": Qk[pcc_bus].X,
            "delta_p": {
                bus: 0.0 if bus == pcc_bus else Pk[bus].X - float(P_ref[bus_position[bus]])
                for bus in nodes
            },
            "delta_q": {
                bus: 0.0 if bus == pcc_bus else Qk[bus].X - float(Q_ref[bus_position[bus]])
                for bus in nodes
            },
        }
    return None


def validate_ac_solution(solution):
    """Rejoue la solution dans Pandapower; les Jacobiennes restent le modele d'optimisation."""
    net = copy.deepcopy(scenario["mv_net"])
    for bus in nodes:
        delta_p = float(solution["delta_p"][bus])
        delta_q = float(solution["delta_q"][bus])
        if abs(delta_p) > 1e-10 or abs(delta_q) > 1e-10:
            pp.create_sgen(net, bus=bus, p_mw=delta_p, q_mvar=delta_q, name="FFOR temporal AC validation")
    try:
        pp.runpp(net, calculate_voltage_angles=True, numba=False, init="auto")
    except Exception:
        return {"valid": False, "reason": "power flow AC non convergent"}
    vm_pu = net.res_bus.reindex(nodes)["vm_pu"]
    loading = net.res_line.loc[net.line["in_service"], "loading_percent"]
    voltage_ok = bool((vm_pu >= delta_Umin - 1e-6).all() and (vm_pu <= delta_Umax + 1e-6).all())
    loading_ok = bool((loading <= 100.0 + 1e-6).all())
    return {
        "valid": voltage_ok and loading_ok,
        "reason": "ok" if voltage_ok and loading_ok else "limite AC depassee",
        "P": float(net.res_ext_grid["p_mw"].sum()),
        "Q": float(net.res_ext_grid["q_mvar"].sum()),
    }


def compute_snapshot_operating_point(scenario_selected, snap):
    """Calcule le point de fonctionnement AC de reference du snapshot."""
    scenario_nodes = list(map(int, scenario_selected["nodes"]))
    scenario_pcc = int(scenario_selected["pcc_bus"])
    bus_position = {bus: index for index, bus in enumerate(scenario_nodes)}
    net = copy.deepcopy(scenario_selected["mv_net"])

    for bus in scenario_nodes:
        if bus == scenario_pcc:
            continue
        target_p = (
            float(snap["P_load"].get(bus, 0.0))
            + float(snap["P_pv_available"].get(bus, 0.0))
            + float(snap["P_hp_max"].get(bus, 0.0))
        )
        target_q = float(snap["Q_load"].get(bus, 0.0))
        delta_p = target_p - float(
            scenario_selected["P_ref"][bus_position[bus]]
        )
        delta_q = target_q - float(
            scenario_selected["Q_ref"][bus_position[bus]]
        )
        if abs(delta_p) > 1e-10 or abs(delta_q) > 1e-10:
            pp.create_sgen(
                net,
                bus=bus,
                p_mw=delta_p,
                q_mvar=delta_q,
                name="FFOR snapshot operating point",
            )

    pp.runpp(net, calculate_voltage_angles=True, numba=False, init="auto")
    vm_pu = net.res_bus.reindex(scenario_nodes)["vm_pu"]
    loading = net.res_line.loc[net.line["in_service"], "loading_percent"]
    if not (
        (vm_pu >= scenario_selected["delta_Umin"] - 1e-6).all()
        and (vm_pu <= scenario_selected["delta_Umax"] + 1e-6).all()
        and (loading <= 100.0 + 1e-6).all()
    ):
        raise RuntimeError(
            "Le point de fonctionnement du snapshot depasse une limite AC."
        )
    return {
        "P": float(net.res_ext_grid["p_mw"].sum()),
        "Q": float(net.res_ext_grid["q_mvar"].sum()),
    }


# ── Balayage sur tous les snapshots ──────────────────────────────────────────
results = {}
operating_points = {}

for (saison, heure), snap in snapshots.items():
    label = f"{saison}_{heure:02d}h"
    print(f"\n⏳ {label} ...")

    operating_points[(saison, heure)] = compute_snapshot_operating_point(
        scenario,
        snap,
    )
    P_pts, Q_pts = [], []
    for i, phi in enumerate(angles):
        a, b = np.cos(phi), np.sin(phi)
        solution = solve_ffor_snapshot(a, b, snap)
        ac_result = validate_ac_solution(solution) if solution is not None else {"valid": False}
        if ac_result["valid"]:
            P_pts.append(ac_result["P"])
            Q_pts.append(ac_result["Q"])

    if P_pts:
        P_pts.append(P_pts[0])
        Q_pts.append(Q_pts[0])
        results[(saison, heure)] = (P_pts, Q_pts)
        print(f"  ✅ P∈[{min(P_pts):.2f},{max(P_pts):.2f}] MW | Q∈[{min(Q_pts):.2f},{max(Q_pts):.2f}] MVAr")
        pd.DataFrame({"P_pcc": P_pts, "Q_pcc": Q_pts}).to_csv(
            output_dir / f"FFOR_{label}.csv", index=False
        )
    else:
        print(f"  ❌ infaisable")

print(f"\n✅ {len(results)}/8 snapshots résolus — résultats dans output/")



## Plots

In [ ]:
# ── Couleurs par heure ────────────────────────────────────────────────────────
COLORS_ETE   = {7: "#FF6B35", 12: "#E63946", 18: "#F4A261", 0: "#457B9D"}
COLORS_HIVER = {7: "#1D3557", 12: "#023E8A", 18: "#0077B6", 0: "#90E0EF"}
HEURE_LABELS = {7: "07h", 12: "12h", 18: "18h", 0: "00h"}

# ── Figure 1 : un panel par saison, 4 heures superposées ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
fig.suptitle("FFOR avec dimension temporelle — Réseau 87_0", fontsize=14, fontweight="bold")

for saison, ax in zip(["Été", "Hiver"], axes):
    colors  = COLORS_ETE if saison == "Été" else COLORS_HIVER
    handles = []
    emoji   = "☀️" if saison == "Été" else "❄️"

    for heure in HEURES:
        if (saison, heure) not in results:
            continue
        P_pts, Q_pts = results[(saison, heure)]
        c = colors[heure]
        ax.plot(P_pts, Q_pts, color=c, linewidth=2, zorder=3)
        ax.fill(P_pts, Q_pts, alpha=0.10, color=c)
        ax.scatter(P_pts[:-1], Q_pts[:-1], s=12, color=c, zorder=4)
        operating_point = operating_points[(saison, heure)]
        ax.scatter(
            [operating_point["P"]],
            [operating_point["Q"]],
            color=c,
            marker="x",
            s=55,
            zorder=5,
        )
        handles.append(mpatches.Patch(color=c, label=HEURE_LABELS[heure]))

    handles.append(
        plt.Line2D(
            [0], [0], marker="x", color="black", linestyle="None",
            label="Point de fonctionnement",
        )
    )

    ax.set_title(f"{emoji} {saison}", fontsize=12)
    ax.set_xlabel("P_pcc (MW)", fontsize=11)
    ax.set_ylabel("Q_pcc (MVAr)", fontsize=11)
    ax.legend(handles=handles, fontsize=9, loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "FFOR_temporal_saisons.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2 : un panel par heure, été vs hiver superposés ───────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 11))
fig2.suptitle("FFOR temporel — Été vs Hiver par heure — Réseau 87_0", fontsize=13, fontweight="bold")

HEURE_FULL = {7: "07h00 — matin", 12: "12h00 — midi", 18: "18h00 — soir", 0: "00h00 — minuit"}

for ax, heure in zip(axes2.flat, HEURES):
    for saison, color in [("Été", "#E63946"), ("Hiver", "#023E8A")]:
        if (saison, heure) not in results:
            continue
        P_pts, Q_pts = results[(saison, heure)]
        emoji = "☀️" if saison == "Été" else "❄️"
        ax.plot(P_pts, Q_pts, color=color, linewidth=2, label=f"{emoji} {saison}")
        ax.fill(P_pts, Q_pts, alpha=0.12, color=color)
        ax.scatter(P_pts[:-1], Q_pts[:-1], s=8, color=color, zorder=4)
        operating_point = operating_points[(saison, heure)]
        ax.scatter(
            [operating_point["P"]],
            [operating_point["Q"]],
            color=color,
            marker="x",
            s=50,
            zorder=5,
        )

    ax.set_title(HEURE_FULL[heure], fontsize=11, fontweight="bold")
    ax.set_xlabel("P_pcc (MW)")
    ax.set_ylabel("Q_pcc (MVAr)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / "FFOR_temporal_heures.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"📊 Figures sauvegardées dans output/")

## FFOR soutenu sur un jour d'ete

Cette cellule calcule le FFOR a une heure precise d'un jour d'ete. Pour garantir une consigne constante pendant toute la duree, elle utilise l'intersection des disponibilites horaires :

- minimum de puissance PV disponible ;
- borne HP la moins negative ;
- reserve thermique HP complete jusqu'a 1 h, puis decroissante lineairement jusqu'a zero a 8 h ;
- point de fonctionnement AC construit avec ces memes disponibilites ;
- intersection cumulative avec tous les FFOR de duree plus courte.

La convention des graphes est `P_flex = P_reference - P_pcc` : une valeur positive signifie davantage d'injection locale, donc moins d'import au PCC. L'intersection cumulative garantit qu'un FFOR long est entierement inclus dans les FFOR plus courts. La loi d'autonomie HP reste l'approximation lineaire initiale : elle doit etre remplacee par des parametres thermiques mesures lorsqu'ils seront disponibles. La puissance HP horaire suit toutefois le profil nPro de demande de chaleur, donc elle varie deja entre ete et hiver. La limite reactive utilisee par le FFOR reste la capacite nominale de l'onduleur et ne depend pas directement de l'irradiance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Parametres modifiables
jour_ete = "2023-07-15"
heure_depart = 14
durees_soutenues = [
    (0.25, "15 min."),
    (0.50, "30 min."),
    (1.00, "1 h"),
    (2.00, "2 h"),
    (4.00, "4 h"),
    (8.00, "8 h"),
]
n_angles_soutenu = 72

# Hypothese lineaire de reserve thermique des pompes a chaleur.
# Jusqu'a 1 h, toute la consommation HP peut etre interrompue. Entre 1 h
# et 8 h, la puissance interruptible diminue lineairement; a 8 h, aucune
# interruption soutenue n'est autorisee.
HP_FULL_FLEX_DURATION_H = 1.0
HP_MAX_FLEX_DURATION_H = 8.0

# Les donnees nPro/PV/HP sont horaires dans ce notebook.
time_index_soutenu = pd.date_range("2023-01-01 00:00", periods=8760, freq="h")
profile_resolution_h = (time_index_soutenu[1] - time_index_soutenu[0]).total_seconds() / 3600
start_ts = pd.Timestamp(f"{jour_ete} {heure_depart:02d}:00")

if start_ts not in time_index_soutenu:
    raise ValueError(f"{start_ts} n'est pas dans l'index temporel 2023.")
if start_ts.month not in ETE_MONTHS:
    print(f"Attention: {start_ts.date()} n'est pas dans les mois d'ete definis: {ETE_MONTHS}")
if profile_resolution_h >= 1:
    print("Note: profils horaires; les durees 15 min et 30 min utilisent la meme disponibilite que l'heure de depart.")

P_pv_dt_df_soutenu = P_pv_max_dt.reset_index().pivot_table(index="time", columns="bus", values="P_pv")
P_hp_dt_df_soutenu = P_hp_max_dt.reset_index().pivot_table(index="time", columns="bus", values="P_hp")

f_load_array_soutenu = np.asarray(f_load_elec)
f_min_soutenu = f_load_array_soutenu.min()
f_max_soutenu = f_load_array_soutenu.max()
alpha_min_soutenu = globals().get("alpha_min", 0.85)
alpha_max_soutenu = globals().get("alpha_max", 1.20)
load_varies_soutenu = FFOR_LOAD_MODE == "temporal"


def indices_pour_duree(start_timestamp, duration_h):
    start_idx = int(time_index_soutenu.get_loc(start_timestamp))
    n_steps = max(1, int(np.ceil(duration_h / profile_resolution_h)))
    end_idx = min(start_idx + n_steps, len(time_index_soutenu))
    return np.arange(start_idx, end_idx)


def hp_sustained_flex_factor(duration_h):
    if duration_h <= HP_FULL_FLEX_DURATION_H:
        return 1.0
    if duration_h >= HP_MAX_FLEX_DURATION_H:
        return 0.0
    return (
        HP_MAX_FLEX_DURATION_H - duration_h
    ) / (
        HP_MAX_FLEX_DURATION_H - HP_FULL_FLEX_DURATION_H
    )


def signed_polygon_area(vertices):
    vertices = np.asarray(vertices, dtype=float)
    x_values = vertices[:, 0]
    y_values = vertices[:, 1]
    return 0.5 * np.sum(
        x_values * np.roll(y_values, -1)
        - y_values * np.roll(x_values, -1)
    )


def convex_polygon(points):
    points = np.unique(np.asarray(points, dtype=float), axis=0)
    if len(points) < 3:
        raise RuntimeError("Le FFOR contient moins de trois points distincts.")
    vertices = points[ConvexHull(points).vertices]
    if signed_polygon_area(vertices) < 0:
        vertices = vertices[::-1]
    return vertices


def line_intersection(segment_start, segment_end, clip_start, clip_end):
    segment_direction = segment_end - segment_start
    clip_direction = clip_end - clip_start
    denominator = np.cross(segment_direction, clip_direction)
    if abs(denominator) < 1e-12:
        return segment_end
    fraction = np.cross(
        clip_start - segment_start,
        clip_direction,
    ) / denominator
    return segment_start + fraction * segment_direction


def intersect_convex_polygons(subject_polygon, clip_polygon):
    # Algorithme de Sutherland-Hodgman. Les deux polygones sont convexes et
    # ordonnes dans le sens anti-horaire par convex_polygon().
    output = [np.asarray(point, dtype=float) for point in subject_polygon]
    clip_polygon = np.asarray(clip_polygon, dtype=float)

    for index, clip_start in enumerate(clip_polygon):
        clip_end = clip_polygon[(index + 1) % len(clip_polygon)]
        input_vertices = output
        output = []
        if not input_vertices:
            break

        segment_start = input_vertices[-1]
        for segment_end in input_vertices:
            end_inside = np.cross(
                clip_end - clip_start,
                segment_end - clip_start,
            ) >= -1e-10
            start_inside = np.cross(
                clip_end - clip_start,
                segment_start - clip_start,
            ) >= -1e-10

            if end_inside:
                if not start_inside:
                    output.append(line_intersection(
                        segment_start,
                        segment_end,
                        clip_start,
                        clip_end,
                    ))
                output.append(segment_end)
            elif start_inside:
                output.append(line_intersection(
                    segment_start,
                    segment_end,
                    clip_start,
                    clip_end,
                ))
            segment_start = segment_end

    if len(output) < 3:
        raise RuntimeError("Intersection vide entre deux FFOR soutenus.")
    return convex_polygon(output)


def build_sustained_snapshot(duration_h):
    idx = indices_pour_duree(start_ts, duration_h)

    # PV: puissance soutenable = minimum de disponibilite sur toute la fenetre.
    P_pv_available_snap = build_pv_available_snapshot_for_scenario(
        scenario,
        idx,
        method="min",
    )

    # HP: P_hp_max est negatif. La borne basse est la consommation de
    # reference soutenable. La borne haute limite son interruption selon la
    # reserve thermique disponible pour la duree demandee.
    P_hp_max_snap = P_hp_dt_df_soutenu.iloc[idx].max().reindex(nodes, fill_value=0.0).to_dict()
    hp_flex_factor = hp_sustained_flex_factor(duration_h)
    P_hp_upper_snap = {
        bus: float(p_hp_min) * (1.0 - hp_flex_factor)
        for bus, p_hp_min in P_hp_max_snap.items()
    }
    Q_pv_max_snap = {
        bus: float(scenario["Q_pv_max"].get(bus, 0.0))
        for bus in nodes
    }

    if load_varies_soutenu:
        f_window = f_load_array_soutenu[idx].mean()
        alpha_window = alpha_min_soutenu + (alpha_max_soutenu - alpha_min_soutenu) * (f_window - f_min_soutenu) / (f_max_soutenu - f_min_soutenu)
        P_load_snap = {bus: P_load.get(bus, 0.0) * alpha_window for bus in nodes}
        Q_load_snap = {bus: Q_load.get(bus, 0.0) * alpha_window for bus in nodes}
    else:
        alpha_window = np.nan
        P_load_snap = {bus: P_load.get(bus, 0.0) for bus in nodes}
        Q_load_snap = {bus: Q_load.get(bus, 0.0) for bus in nodes}

    return {
        "P_pv_available": P_pv_available_snap,
        "Q_pv_max": Q_pv_max_snap,
        "P_hp_max": P_hp_max_snap,
        "P_hp_upper": P_hp_upper_snap,
        "hp_flex_factor": hp_flex_factor,
        "P_load": P_load_snap,
        "Q_load": Q_load_snap,
        "alpha": alpha_window,
        "time_indices": idx,
    }


angles_soutenu = np.linspace(0, 2 * np.pi, n_angles_soutenu, endpoint=False)
results_soutenus = {}
summary_soutenu = []
previous_sustained_polygon = None

for duration_h, duration_label in durees_soutenues:
    snap = build_sustained_snapshot(duration_h)
    print(f"\nFFOR soutenu {duration_label} depuis {start_ts:%Y-%m-%d %H:%M} ...")

    operating_point = compute_snapshot_operating_point(scenario, snap)
    P_pts, Q_pts = [], []
    for phi in angles_soutenu:
        solution = solve_ffor_snapshot(np.cos(phi), np.sin(phi), snap)
        ac_result = validate_ac_solution(solution) if solution is not None else {"valid": False}
        if ac_result["valid"]:
            P_pts.append(ac_result["P"])
            Q_pts.append(ac_result["Q"])

    if len(P_pts) < 3:
        print("  infaisable ou pas assez de points AC valides")
        continue

    # Convention reseau: positif = plus d'injection locale / moins d'import.
    raw_P_flex = [operating_point["P"] - p for p in P_pts]
    raw_Q_flex = [operating_point["Q"] - q for q in Q_pts]
    raw_polygon = convex_polygon(np.column_stack([
        raw_P_flex,
        raw_Q_flex,
    ]))

    # Une flexibilite soutenue D heures doit etre realisable sur toute la
    # fenetre. Le domaine est donc l'intersection cumulative des FFOR de
    # durees croissantes, ce qui garantit leur emboitement.
    if previous_sustained_polygon is None:
        sustained_polygon = raw_polygon
    else:
        sustained_polygon = intersect_convex_polygons(
            previous_sustained_polygon,
            raw_polygon,
        )
    previous_sustained_polygon = sustained_polygon

    P_flex = np.append(
        sustained_polygon[:, 0],
        sustained_polygon[0, 0],
    ).tolist()
    Q_flex = np.append(
        sustained_polygon[:, 1],
        sustained_polygon[0, 1],
    ).tolist()
    P_pts = [operating_point["P"] - value for value in P_flex]
    Q_pts = [operating_point["Q"] - value for value in Q_flex]

    results_soutenus[duration_h] = {
        "label": duration_label,
        "P_pcc": P_pts,
        "Q_pcc": Q_pts,
        "P_flex": P_flex,
        "Q_flex": Q_flex,
        "P_flex_pos_max": max(P_flex),
        "P_flex_neg_min": min(P_flex),
    }

    summary_soutenu.append({
        "duration_h": duration_h,
        "duration_label": duration_label,
        "P_flex_pos_max_MW": max(P_flex),
        "P_flex_neg_min_MW": min(P_flex),
        "P_flex_pos_raw_MW": max(raw_P_flex),
        "P_flex_neg_raw_MW": min(raw_P_flex),
        "Q_flex_max_MVAr": max(Q_flex),
        "Q_flex_min_MVAr": min(Q_flex),
        "P_reference_MW": operating_point["P"],
        "Q_reference_MVAr": operating_point["Q"],
        "hp_flex_factor": snap["hp_flex_factor"],
        "P_hp_reference_MW": sum(snap["P_hp_max"].values()),
        "P_hp_upper_MW": sum(snap["P_hp_upper"].values()),
        "alpha": snap["alpha"],
        "n_time_steps": len(snap["time_indices"]),
    })

    file_label = f"{start_ts:%Y%m%d_%Hh}_{duration_label.replace(' ', '').replace('.', '').replace('min', 'min')}"
    pd.DataFrame({
        "P_pcc_MW": P_pts,
        "Q_pcc_MVAr": Q_pts,
        "P_flex_MW": P_flex,
        "Q_flex_MVAr": Q_flex,
    }).to_csv(output_dir / f"FFOR_sustained_{file_label}.csv", index=False)
    print(f"  Pflex=[{min(P_flex):.2f}, {max(P_flex):.2f}] MW | Qflex=[{min(Q_flex):.2f}, {max(Q_flex):.2f}] MVAr")

if not results_soutenus:
    raise RuntimeError("Aucun FFOR soutenu n'a pu etre calcule.")

summary_soutenu_df = pd.DataFrame(summary_soutenu)
summary_soutenu_df.to_csv(output_dir / f"FFOR_sustained_summary_{start_ts:%Y%m%d_%Hh}.csv", index=False)

# Figure type article: FFOR P-Q par duree + enveloppe active positive/negative.
duration_colors = ["#DBEAFE", "#BFDBFE", "#93C5FD", "#60A5FA", "#2563EB", "#0B3B75"]
fig, (ax_ffor, ax_power) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(f"FFOR soutenu - {start_ts:%d.%m.%Y %H:%M} - {scenario['name']}", fontsize=13, fontweight="bold")

for (duration_h, duration_label), color in zip(durees_soutenues, duration_colors):
    if duration_h not in results_soutenus:
        continue
    data = results_soutenus[duration_h]
    ax_ffor.fill(data["P_flex"], data["Q_flex"], facecolor=color, edgecolor="#0B3B75", linewidth=1.5, alpha=0.72, label=duration_label)

ax_ffor.axhline(0, color="black", linewidth=0.8, alpha=0.45)
ax_ffor.axvline(0, color="black", linewidth=0.8, alpha=0.45)
ax_ffor.set_xlabel("Active Power Flexibility [MW]")
ax_ffor.set_ylabel("Reactive Power Flexibility [MVAr]")
ax_ffor.grid(True, alpha=0.35, linestyle="--")
ax_ffor.legend(ncol=2, loc="upper left", frameon=True, fancybox=False, edgecolor="black")

valid_durations = [duration_h for duration_h, _ in durees_soutenues if duration_h in results_soutenus]
valid_labels = [label for duration_h, label in durees_soutenues if duration_h in results_soutenus]
positive_flex = [results_soutenus[duration_h]["P_flex_pos_max"] for duration_h in valid_durations]
negative_flex = [results_soutenus[duration_h]["P_flex_neg_min"] for duration_h in valid_durations]

ax_power.fill_between(valid_durations, 0, positive_flex, color="#94A3B8", alpha=0.65, label="Positive Flexibility")
ax_power.fill_between(valid_durations, negative_flex, 0, color="#BFDBFE", alpha=0.75, label="Negative Flexibility")
ax_power.plot(valid_durations, positive_flex, color="#0B3B75", linewidth=2)
ax_power.plot(valid_durations, negative_flex, color="#0284C7", linewidth=2)
ax_power.axhline(0, color="black", linewidth=0.8, alpha=0.55)
ax_power.set_xlabel("Sustained Duration [h]")
ax_power.set_ylabel("Active Power Flexibility [MW]")
display_tick_durations = [0.25, 1.0, 2.0, 4.0, 8.0]
display_tick_labels = ["15 min", "1 h", "2 h", "4 h", "8 h"]
ax_power.set_xticks(display_tick_durations)
ax_power.set_xticklabels(display_tick_labels)
ax_power.grid(True, alpha=0.35, linestyle="--")
ax_power.legend(loc="upper right", frameon=True, fancybox=False, edgecolor="black")

plt.tight_layout()
plt.savefig(output_dir / f"FFOR_sustained_{start_ts:%Y%m%d_%Hh}.png", dpi=150, bbox_inches="tight")
plt.show()

display(summary_soutenu_df)
print(f"Figure et CSV sauvegardes dans {output_dir}/")


## FFOR soutenu avec BelalpSolar

Cette cellule applique exactement les memes contraintes et la meme convention de signe au scenario `with_belalp`. La disponibilite active de BelalpSolar suit le profil d'irradiance du bus de raccordement le plus proche.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp
import gurobipy as gp
from gurobipy import GRB
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

scenario_soutenu_belalp = scenarios["with_belalp"]
nodes_belalp = scenario_soutenu_belalp["nodes"]
pcc_bus_belalp = scenario_soutenu_belalp["pcc_bus"]
P_base_belalp = float(scenario_soutenu_belalp["P_base"])
Q_base_belalp = float(scenario_soutenu_belalp["Q_base"])

# Memes parametres que la cellule sans BelalpSolar.
jour_ete_belalp = globals().get("jour_ete", "2023-07-15")
heure_depart_belalp = globals().get("heure_depart", 14)
durees_soutenues_belalp = globals().get("durees_soutenues", [
    (0.25, "15 min."),
    (0.50, "30 min."),
    (1.00, "1 h"),
    (2.00, "2 h"),
    (4.00, "4 h"),
    (8.00, "8 h"),
])
n_angles_soutenu_belalp = globals().get("n_angles_soutenu", 72)

time_index_soutenu_belalp = pd.date_range("2023-01-01 00:00", periods=8760, freq="h")
profile_resolution_h_belalp = (time_index_soutenu_belalp[1] - time_index_soutenu_belalp[0]).total_seconds() / 3600
start_ts_belalp = pd.Timestamp(f"{jour_ete_belalp} {heure_depart_belalp:02d}:00")

if start_ts_belalp not in time_index_soutenu_belalp:
    raise ValueError(f"{start_ts_belalp} n'est pas dans l'index temporel 2023.")
if start_ts_belalp.month not in ETE_MONTHS:
    print(f"Attention: {start_ts_belalp.date()} n'est pas dans les mois d'ete definis: {ETE_MONTHS}")
if profile_resolution_h_belalp >= 1:
    print("Note: profils horaires; les durees 15 min et 30 min utilisent la meme disponibilite que l'heure de depart.")

P_pv_dt_df_belalp = P_pv_max_dt.reset_index().pivot_table(index="time", columns="bus", values="P_pv")
P_hp_dt_df_belalp = P_hp_max_dt.reset_index().pivot_table(index="time", columns="bus", values="P_hp")

f_load_array_belalp = np.asarray(f_load_elec)
f_min_belalp = f_load_array_belalp.min()
f_max_belalp = f_load_array_belalp.max()
alpha_min_belalp = globals().get("alpha_min", 0.85)
alpha_max_belalp = globals().get("alpha_max", 1.20)
load_varies_belalp = FFOR_LOAD_MODE == "temporal"


def indices_pour_duree_belalp(start_timestamp, duration_h):
    start_idx = int(time_index_soutenu_belalp.get_loc(start_timestamp))
    n_steps = max(1, int(np.ceil(duration_h / profile_resolution_h_belalp)))
    end_idx = min(start_idx + n_steps, len(time_index_soutenu_belalp))
    return np.arange(start_idx, end_idx)


def build_sustained_snapshot_belalp(duration_h):
    idx = indices_pour_duree_belalp(start_ts_belalp, duration_h)

    P_pv_available_snap = build_pv_available_snapshot_for_scenario(
        scenario_soutenu_belalp,
        idx,
        method="min",
    )

    P_hp_max_snap = P_hp_dt_df_belalp.iloc[idx].max().reindex(nodes_belalp, fill_value=0.0).to_dict()
    hp_flex_factor = hp_sustained_flex_factor(duration_h)
    P_hp_upper_snap = {
        bus: float(p_hp_min) * (1.0 - hp_flex_factor)
        for bus, p_hp_min in P_hp_max_snap.items()
    }
    Q_pv_max_snap = {
        bus: float(scenario_soutenu_belalp["Q_pv_max"].get(bus, 0.0))
        for bus in nodes_belalp
    }

    if load_varies_belalp:
        f_window = f_load_array_belalp[idx].mean()
        alpha_window = alpha_min_belalp + (alpha_max_belalp - alpha_min_belalp) * (f_window - f_min_belalp) / (f_max_belalp - f_min_belalp)
        P_load_snap = {bus: scenario_soutenu_belalp["P_load"].get(bus, 0.0) * alpha_window for bus in nodes_belalp}
        Q_load_snap = {bus: scenario_soutenu_belalp["Q_load"].get(bus, 0.0) * alpha_window for bus in nodes_belalp}
    else:
        alpha_window = np.nan
        P_load_snap = {bus: scenario_soutenu_belalp["P_load"].get(bus, 0.0) for bus in nodes_belalp}
        Q_load_snap = {bus: scenario_soutenu_belalp["Q_load"].get(bus, 0.0) for bus in nodes_belalp}

    return {
        "P_pv_available": P_pv_available_snap,
        "Q_pv_max": Q_pv_max_snap,
        "P_hp_max": P_hp_max_snap,
        "P_hp_upper": P_hp_upper_snap,
        "hp_flex_factor": hp_flex_factor,
        "P_load": P_load_snap,
        "Q_load": Q_load_snap,
        "alpha": alpha_window,
        "time_indices": idx,
    }


def solve_ffor_snapshot_belalp(a, b, snap):
    bus_position = {bus: index for index, bus in enumerate(nodes_belalp)}
    model = gp.Model("FFOR_sustained_with_belalp")
    model.Params.OutputFlag = 0
    model.Params.DualReductions = 0

    Pk = model.addVars(nodes_belalp, lb=-GRB.INFINITY, name="Pk")
    Qk = model.addVars(nodes_belalp, lb=-GRB.INFINITY, name="Qk")
    theta = model.addVars(nodes_belalp, lb=-GRB.INFINITY, name="theta")
    delta_U = model.addVars(nodes_belalp, lb=-GRB.INFINITY, name="U")
    P_pv = model.addVars(nodes_belalp, lb=0.0, name="Ppv")
    Q_pv = model.addVars(nodes_belalp, lb=-GRB.INFINITY, name="Qpv")
    P_hp = model.addVars(
        nodes_belalp,
        lb=-GRB.INFINITY,
        ub=0.0,
        name="Php",
    )

    for bus in nodes_belalp:
        position = bus_position[bus]
        if bus != pcc_bus_belalp:
            model.addConstr(Pk[bus] == P_pv[bus] + P_hp[bus] + snap["P_load"].get(bus, 0.0))
            model.addConstr(Qk[bus] == Q_pv[bus] + snap["Q_load"].get(bus, 0.0))

        model.addConstr(Pk[bus] == scenario_soutenu_belalp["P_ref"][position] + gp.quicksum(
            scenario_soutenu_belalp["J_Ptheta"][position, other_position] * theta[other_bus]
            + scenario_soutenu_belalp["J_PU"][position, other_position] * delta_U[other_bus]
            for other_position, other_bus in enumerate(nodes_belalp)
        ))
        model.addConstr(Qk[bus] == scenario_soutenu_belalp["Q_ref"][position] + gp.quicksum(
            scenario_soutenu_belalp["J_Qtheta"][position, other_position] * theta[other_bus]
            + scenario_soutenu_belalp["J_QU"][position, other_position] * delta_U[other_bus]
            for other_position, other_bus in enumerate(nodes_belalp)
        ))

        model.addConstr(P_pv[bus] <= snap["P_pv_available"].get(bus, 0.0))
        model.addConstr(Q_pv[bus] <= snap["Q_pv_max"].get(bus, 0.0))
        model.addConstr(Q_pv[bus] >= -snap["Q_pv_max"].get(bus, 0.0))
        model.addConstr(P_hp[bus] >= snap["P_hp_max"].get(bus, 0.0))
        model.addConstr(
            P_hp[bus] <= snap.get("P_hp_upper", {}).get(bus, 0.0)
        )

        inverter_smax = float(scenario_soutenu_belalp["S_inv_max"].get(bus, 0.0))
        if inverter_smax > 1e-9:
            model.addQConstr(P_pv[bus] * P_pv[bus] + Q_pv[bus] * Q_pv[bus] <= inverter_smax * inverter_smax)
        else:
            model.addConstr(P_pv[bus] == 0.0)
            model.addConstr(Q_pv[bus] == 0.0)

        model.addConstr(scenario_soutenu_belalp["V"] + delta_U[bus] >= scenario_soutenu_belalp["delta_Umin"] * scenario_soutenu_belalp["V"])
        model.addConstr(scenario_soutenu_belalp["V"] + delta_U[bus] <= scenario_soutenu_belalp["delta_Umax"] * scenario_soutenu_belalp["V"])

    for _, row in scenario_soutenu_belalp["line_data"].iterrows():
        from_bus = int(row["from_bus"])
        to_bus = int(row["to_bus"])
        angle_delta = theta[from_bus] - theta[to_bus]
        voltage_delta = delta_U[from_bus] - delta_U[to_bus]
        Pij = (
            float(row["P_base"])
            - float(row["b"]) * angle_delta
            + float(row["g"]) * voltage_delta
        )
        Qij = (
            float(row["Q_base"])
            - float(row["g"]) * angle_delta
            - float(row["b"]) * voltage_delta
        )
        model.addQConstr(Pij * Pij + Qij * Qij <= float(row["S_max"]) ** 2)

    model.addConstr(theta[pcc_bus_belalp] == 0.0)
    model.addConstr(delta_U[pcc_bus_belalp] == 0.0)
    model.setObjective(a * Pk[pcc_bus_belalp] + b * Qk[pcc_bus_belalp], GRB.MINIMIZE)
    model.optimize()

    if model.status != GRB.OPTIMAL:
        model.dispose()
        return None

    result = {
        "P_linear": Pk[pcc_bus_belalp].X,
        "Q_linear": Qk[pcc_bus_belalp].X,
        "delta_p": {
            bus: 0.0 if bus == pcc_bus_belalp else Pk[bus].X - float(scenario_soutenu_belalp["P_ref"][bus_position[bus]])
            for bus in nodes_belalp
        },
        "delta_q": {
            bus: 0.0 if bus == pcc_bus_belalp else Qk[bus].X - float(scenario_soutenu_belalp["Q_ref"][bus_position[bus]])
            for bus in nodes_belalp
        },
    }
    model.dispose()
    return result


def validate_ac_solution_belalp(solution):
    net = copy.deepcopy(scenario_soutenu_belalp["mv_net"])
    for bus in nodes_belalp:
        delta_p = float(solution["delta_p"][bus])
        delta_q = float(solution["delta_q"][bus])
        if abs(delta_p) > 1e-10 or abs(delta_q) > 1e-10:
            pp.create_sgen(net, bus=bus, p_mw=delta_p, q_mvar=delta_q, name="FFOR sustained Belalp AC validation")
    try:
        pp.runpp(net, calculate_voltage_angles=True, numba=False, init="auto")
    except Exception:
        return {"valid": False, "reason": "power flow AC non convergent"}

    vm_pu = net.res_bus.reindex(nodes_belalp)["vm_pu"]
    loading = net.res_line.loc[net.line["in_service"], "loading_percent"]
    voltage_ok = bool((vm_pu >= scenario_soutenu_belalp["delta_Umin"] - 1e-6).all() and (vm_pu <= scenario_soutenu_belalp["delta_Umax"] + 1e-6).all())
    loading_ok = bool((loading <= 100.0 + 1e-6).all())
    return {
        "valid": voltage_ok and loading_ok,
        "reason": "ok" if voltage_ok and loading_ok else "limite AC depassee",
        "P": float(net.res_ext_grid["p_mw"].sum()),
        "Q": float(net.res_ext_grid["q_mvar"].sum()),
    }


angles_soutenu_belalp = np.linspace(0, 2 * np.pi, n_angles_soutenu_belalp, endpoint=False)
results_soutenus_belalp = {}
summary_soutenu_belalp = []
previous_sustained_polygon_belalp = None

for duration_h, duration_label in durees_soutenues_belalp:
    snap = build_sustained_snapshot_belalp(duration_h)
    print(f"\nFFOR soutenu avec BelalpSolar {duration_label} depuis {start_ts_belalp:%Y-%m-%d %H:%M} ...")

    operating_point = compute_snapshot_operating_point(
        scenario_soutenu_belalp,
        snap,
    )
    P_pts, Q_pts = [], []
    for phi in angles_soutenu_belalp:
        solution = solve_ffor_snapshot_belalp(np.cos(phi), np.sin(phi), snap)
        ac_result = validate_ac_solution_belalp(solution) if solution is not None else {"valid": False}
        if ac_result["valid"]:
            P_pts.append(ac_result["P"])
            Q_pts.append(ac_result["Q"])

    if len(P_pts) < 3:
        print("  infaisable ou pas assez de points AC valides")
        continue

    raw_P_flex = [operating_point["P"] - p for p in P_pts]
    raw_Q_flex = [operating_point["Q"] - q for q in Q_pts]
    raw_polygon = convex_polygon(np.column_stack([
        raw_P_flex,
        raw_Q_flex,
    ]))

    if previous_sustained_polygon_belalp is None:
        sustained_polygon = raw_polygon
    else:
        sustained_polygon = intersect_convex_polygons(
            previous_sustained_polygon_belalp,
            raw_polygon,
        )
    previous_sustained_polygon_belalp = sustained_polygon

    P_flex = np.append(
        sustained_polygon[:, 0],
        sustained_polygon[0, 0],
    ).tolist()
    Q_flex = np.append(
        sustained_polygon[:, 1],
        sustained_polygon[0, 1],
    ).tolist()
    P_pts = [operating_point["P"] - value for value in P_flex]
    Q_pts = [operating_point["Q"] - value for value in Q_flex]

    results_soutenus_belalp[duration_h] = {
        "label": duration_label,
        "P_pcc": P_pts,
        "Q_pcc": Q_pts,
        "P_flex": P_flex,
        "Q_flex": Q_flex,
        "P_flex_pos_max": max(P_flex),
        "P_flex_neg_min": min(P_flex),
    }

    summary_soutenu_belalp.append({
        "duration_h": duration_h,
        "duration_label": duration_label,
        "P_flex_pos_max_MW": max(P_flex),
        "P_flex_neg_min_MW": min(P_flex),
        "P_flex_pos_raw_MW": max(raw_P_flex),
        "P_flex_neg_raw_MW": min(raw_P_flex),
        "Q_flex_max_MVAr": max(Q_flex),
        "Q_flex_min_MVAr": min(Q_flex),
        "P_reference_MW": operating_point["P"],
        "Q_reference_MVAr": operating_point["Q"],
        "hp_flex_factor": snap["hp_flex_factor"],
        "P_hp_reference_MW": sum(snap["P_hp_max"].values()),
        "P_hp_upper_MW": sum(snap["P_hp_upper"].values()),
        "alpha": snap["alpha"],
        "n_time_steps": len(snap["time_indices"]),
    })

    file_label = f"with_belalp_{start_ts_belalp:%Y%m%d_%Hh}_{duration_label.replace(' ', '').replace('.', '').replace('min', 'min')}"
    pd.DataFrame({
        "P_pcc_MW": P_pts,
        "Q_pcc_MVAr": Q_pts,
        "P_flex_MW": P_flex,
        "Q_flex_MVAr": Q_flex,
    }).to_csv(output_dir / f"FFOR_sustained_{file_label}.csv", index=False)
    print(f"  Pflex=[{min(P_flex):.2f}, {max(P_flex):.2f}] MW | Qflex=[{min(Q_flex):.2f}, {max(Q_flex):.2f}] MVAr")

if not results_soutenus_belalp:
    raise RuntimeError("Aucun FFOR soutenu avec BelalpSolar n'a pu etre calcule.")

summary_soutenu_belalp_df = pd.DataFrame(summary_soutenu_belalp)
summary_soutenu_belalp_df.to_csv(output_dir / f"FFOR_sustained_with_belalp_summary_{start_ts_belalp:%Y%m%d_%Hh}.csv", index=False)

duration_colors_belalp = ["#DCFCE7", "#BBF7D0", "#86EFAC", "#4ADE80", "#16A34A", "#14532D"]
fig, (ax_ffor, ax_power) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(f"FFOR soutenu avec BelalpSolar - {start_ts_belalp:%d.%m.%Y %H:%M} - {scenario_soutenu_belalp['name']}", fontsize=13, fontweight="bold")

for (duration_h, duration_label), color in zip(durees_soutenues_belalp, duration_colors_belalp):
    if duration_h not in results_soutenus_belalp:
        continue
    data = results_soutenus_belalp[duration_h]
    ax_ffor.fill(data["P_flex"], data["Q_flex"], facecolor=color, edgecolor="#14532D", linewidth=1.5, alpha=0.72, label=duration_label)

ax_ffor.axhline(0, color="black", linewidth=0.8, alpha=0.45)
ax_ffor.axvline(0, color="black", linewidth=0.8, alpha=0.45)
ax_ffor.set_xlabel("Active Power Flexibility [MW]")
ax_ffor.set_ylabel("Reactive Power Flexibility [MVAr]")
ax_ffor.grid(True, alpha=0.35, linestyle="--")
ax_ffor.legend(ncol=2, loc="upper left", frameon=True, fancybox=False, edgecolor="black")

valid_durations_belalp = [duration_h for duration_h, _ in durees_soutenues_belalp if duration_h in results_soutenus_belalp]
valid_labels_belalp = [label for duration_h, label in durees_soutenues_belalp if duration_h in results_soutenus_belalp]
positive_flex_belalp = [results_soutenus_belalp[duration_h]["P_flex_pos_max"] for duration_h in valid_durations_belalp]
negative_flex_belalp = [results_soutenus_belalp[duration_h]["P_flex_neg_min"] for duration_h in valid_durations_belalp]

ax_power.fill_between(valid_durations_belalp, 0, positive_flex_belalp, color="#86EFAC", alpha=0.65, label="Positive Flexibility")
ax_power.fill_between(valid_durations_belalp, negative_flex_belalp, 0, color="#DCFCE7", alpha=0.85, label="Negative Flexibility")
ax_power.plot(valid_durations_belalp, positive_flex_belalp, color="#14532D", linewidth=2)
ax_power.plot(valid_durations_belalp, negative_flex_belalp, color="#15803D", linewidth=2)
ax_power.axhline(0, color="black", linewidth=0.8, alpha=0.55)
ax_power.set_xlabel("Sustained Duration [h]")
ax_power.set_ylabel("Active Power Flexibility [MW]")
ax_power.set_xticks(display_tick_durations)
ax_power.set_xticklabels(display_tick_labels)
ax_power.grid(True, alpha=0.35, linestyle="--")
ax_power.legend(loc="upper right", frameon=True, fancybox=False, edgecolor="black")

plt.tight_layout()
plt.savefig(output_dir / f"FFOR_sustained_with_belalp_{start_ts_belalp:%Y%m%d_%Hh}.png", dpi=150, bbox_inches="tight")
plt.show()

display(summary_soutenu_belalp_df)
print(f"Figure et CSV avec BelalpSolar sauvegardes dans {output_dir}/")


In [ ]:
# FFOR temporel avec BelalpSolar, en gardant les resultats sans BelalpSolar
scenario_with_belalp = scenarios["with_belalp"]

snapshots_with_belalp = build_snapshots_for_scenario(
    scenario_with_belalp,
    load_mode=FFOR_LOAD_MODE,
)


def solve_ffor_snapshot_for_scenario(scenario_selected, a, b, snap):
    scenario_nodes = scenario_selected["nodes"]
    scenario_pcc_bus = scenario_selected["pcc_bus"]
    bus_position = {bus: index for index, bus in enumerate(scenario_nodes)}

    model = gp.Model(f"FFOR_temporal_{scenario_selected['name']}")
    model.Params.OutputFlag = 0
    model.Params.DualReductions = 0

    Pk = model.addVars(scenario_nodes, lb=-GRB.INFINITY, name="Pk")
    Qk = model.addVars(scenario_nodes, lb=-GRB.INFINITY, name="Qk")
    theta = model.addVars(scenario_nodes, lb=-GRB.INFINITY, name="theta")
    delta_U = model.addVars(scenario_nodes, lb=-GRB.INFINITY, name="U")
    P_pv = model.addVars(scenario_nodes, lb=0.0, name="Ppv")
    Q_pv = model.addVars(scenario_nodes, lb=-GRB.INFINITY, name="Qpv")
    P_hp = model.addVars(
        scenario_nodes,
        lb=-GRB.INFINITY,
        ub=0.0,
        name="Php",
    )

    for bus in scenario_nodes:
        position = bus_position[bus]
        if bus != scenario_pcc_bus:
            model.addConstr(Pk[bus] == P_pv[bus] + P_hp[bus] + snap["P_load"].get(bus, 0.0))
            model.addConstr(Qk[bus] == Q_pv[bus] + snap["Q_load"].get(bus, 0.0))

        model.addConstr(Pk[bus] == scenario_selected["P_ref"][position] + gp.quicksum(
            scenario_selected["J_Ptheta"][position, other_position] * theta[other_bus]
            + scenario_selected["J_PU"][position, other_position] * delta_U[other_bus]
            for other_position, other_bus in enumerate(scenario_nodes)
        ))
        model.addConstr(Qk[bus] == scenario_selected["Q_ref"][position] + gp.quicksum(
            scenario_selected["J_Qtheta"][position, other_position] * theta[other_bus]
            + scenario_selected["J_QU"][position, other_position] * delta_U[other_bus]
            for other_position, other_bus in enumerate(scenario_nodes)
        ))

        model.addConstr(P_pv[bus] <= snap["P_pv_available"].get(bus, 0.0))
        model.addConstr(Q_pv[bus] <= snap["Q_pv_max"].get(bus, 0.0))
        model.addConstr(Q_pv[bus] >= -snap["Q_pv_max"].get(bus, 0.0))
        model.addConstr(P_hp[bus] >= snap["P_hp_max"].get(bus, 0.0))

        inverter_smax = float(scenario_selected["S_inv_max"].get(bus, 0.0))
        if inverter_smax > 1e-9:
            model.addQConstr(P_pv[bus] * P_pv[bus] + Q_pv[bus] * Q_pv[bus] <= inverter_smax * inverter_smax)
        else:
            model.addConstr(P_pv[bus] == 0.0)
            model.addConstr(Q_pv[bus] == 0.0)

        model.addConstr(scenario_selected["V"] + delta_U[bus] >= scenario_selected["delta_Umin"] * scenario_selected["V"])
        model.addConstr(scenario_selected["V"] + delta_U[bus] <= scenario_selected["delta_Umax"] * scenario_selected["V"])

    for _, row in scenario_selected["line_data"].iterrows():
        from_bus = int(row["from_bus"])
        to_bus = int(row["to_bus"])
        angle_delta = theta[from_bus] - theta[to_bus]
        voltage_delta = delta_U[from_bus] - delta_U[to_bus]
        Pij = (
            float(row["P_base"])
            - float(row["b"]) * angle_delta
            + float(row["g"]) * voltage_delta
        )
        Qij = (
            float(row["Q_base"])
            - float(row["g"]) * angle_delta
            - float(row["b"]) * voltage_delta
        )
        model.addQConstr(Pij * Pij + Qij * Qij <= float(row["S_max"]) ** 2)

    model.addConstr(theta[scenario_pcc_bus] == 0.0)
    model.addConstr(delta_U[scenario_pcc_bus] == 0.0)
    model.setObjective(a * Pk[scenario_pcc_bus] + b * Qk[scenario_pcc_bus], GRB.MINIMIZE)
    model.optimize()

    if model.status != GRB.OPTIMAL:
        model.dispose()
        return None

    result = {
        "P_linear": Pk[scenario_pcc_bus].X,
        "Q_linear": Qk[scenario_pcc_bus].X,
        "delta_p": {
            bus: 0.0 if bus == scenario_pcc_bus else Pk[bus].X - float(scenario_selected["P_ref"][bus_position[bus]])
            for bus in scenario_nodes
        },
        "delta_q": {
            bus: 0.0 if bus == scenario_pcc_bus else Qk[bus].X - float(scenario_selected["Q_ref"][bus_position[bus]])
            for bus in scenario_nodes
        },
    }
    model.dispose()
    return result


def validate_ac_solution_for_scenario(scenario_selected, solution):
    scenario_nodes = scenario_selected["nodes"]
    net = copy.deepcopy(scenario_selected["mv_net"])
    for bus in scenario_nodes:
        delta_p = float(solution["delta_p"][bus])
        delta_q = float(solution["delta_q"][bus])
        if abs(delta_p) > 1e-10 or abs(delta_q) > 1e-10:
            pp.create_sgen(net, bus=bus, p_mw=delta_p, q_mvar=delta_q, name="FFOR temporal AC validation")
    try:
        pp.runpp(net, calculate_voltage_angles=True, numba=False, init="auto")
    except Exception:
        return {"valid": False, "reason": "power flow AC non convergent"}

    vm_pu = net.res_bus.reindex(scenario_nodes)["vm_pu"]
    loading = net.res_line.loc[net.line["in_service"], "loading_percent"]
    voltage_ok = bool((vm_pu >= scenario_selected["delta_Umin"] - 1e-6).all() and (vm_pu <= scenario_selected["delta_Umax"] + 1e-6).all())
    loading_ok = bool((loading <= 100.0 + 1e-6).all())
    return {
        "valid": voltage_ok and loading_ok,
        "reason": "ok" if voltage_ok and loading_ok else "limite AC depassee",
        "P": float(net.res_ext_grid["p_mw"].sum()),
        "Q": float(net.res_ext_grid["q_mvar"].sum()),
    }


results_with_belalp = {}
operating_points_with_belalp = {}

for (saison, heure), snap in snapshots_with_belalp.items():
    label = f"{saison}_{heure:02d}h"
    print(f"\nAvec BelalpSolar - {label} ...")
    operating_points_with_belalp[(saison, heure)] = (
        compute_snapshot_operating_point(scenario_with_belalp, snap)
    )
    P_pts, Q_pts = [], []

    for phi in angles:
        solution = solve_ffor_snapshot_for_scenario(scenario_with_belalp, np.cos(phi), np.sin(phi), snap)
        ac_result = validate_ac_solution_for_scenario(scenario_with_belalp, solution) if solution is not None else {"valid": False}
        if ac_result["valid"]:
            P_pts.append(ac_result["P"])
            Q_pts.append(ac_result["Q"])

    if P_pts:
        P_pts.append(P_pts[0])
        Q_pts.append(Q_pts[0])
        results_with_belalp[(saison, heure)] = (P_pts, Q_pts)
        pd.DataFrame({"P_pcc": P_pts, "Q_pcc": Q_pts}).to_csv(output_dir / f"FFOR_with_belalp_{label}.csv", index=False)
        print(f"  P=[{min(P_pts):.2f},{max(P_pts):.2f}] MW | Q=[{min(Q_pts):.2f},{max(Q_pts):.2f}] MVAr")
    else:
        print("  infaisable")

fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharex=False, sharey=False)
fig.suptitle("FFOR temporel - sans vs avec BelalpSolar", fontsize=14, fontweight="bold")

for ax, saison in zip(axes[:, 0], ["Été", "Hiver"]):
    ax.set_ylabel(f"{saison}\nQ_pcc (MVAr)")

for row, saison in enumerate(["Été", "Hiver"]):
    for col, heure in enumerate(HEURES):
        ax = axes[row, col]
        key = (saison, heure)
        if key in results:
            P_without, Q_without = results[key]
            ax.plot(P_without, Q_without, color="#1D4ED8", linewidth=2, label="Sans BelalpSolar")
            ax.fill(P_without, Q_without, color="#1D4ED8", alpha=0.10)
            operating_without = operating_points[key]
            ax.scatter(
                [operating_without["P"]], [operating_without["Q"]],
                color="#1D4ED8", marker="x", s=45,
            )
        if key in results_with_belalp:
            P_with, Q_with = results_with_belalp[key]
            ax.plot(P_with, Q_with, color="#15803D", linewidth=2, label="Avec BelalpSolar")
            ax.fill(P_with, Q_with, color="#15803D", alpha=0.10)
            operating_with = operating_points_with_belalp[key]
            ax.scatter(
                [operating_with["P"]], [operating_with["Q"]],
                color="#15803D", marker="x", s=45,
            )
        ax.set_title(f"{saison} - {heure:02d}h")
        ax.set_xlabel("P_pcc (MW)")
        ax.grid(True, alpha=0.3)
        if row == 0 and col == 0:
            ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(output_dir / "FFOR_temporal_without_vs_with_belalp.png", dpi=150, bbox_inches="tight")
plt.show()

for key in sorted(results_with_belalp):
    P_without, Q_without = results.get(key, ([np.nan], [np.nan]))
    P_with, Q_with = results_with_belalp[key]
    print(
        f"{key}: sans P=[{np.nanmin(P_without):.2f},{np.nanmax(P_without):.2f}], "
        f"avec P=[{min(P_with):.2f},{max(P_with):.2f}] MW | "
        f"sans Q=[{np.nanmin(Q_without):.2f},{np.nanmax(Q_without):.2f}], "
        f"avec Q=[{min(Q_with):.2f},{max(Q_with):.2f}] MVAr"
    )

print("Metadonnees BelalpSolar:", scenario_with_belalp.get("metadata", {}))
print(f"Resultats avec BelalpSolar sauvegardes dans {output_dir}/")


In [ ]:
# Estimation du temps
import time

# Test sur 1 snapshot
snap_test = snapshots[("Été", 12)]
start = time.time()
solve_ffor_snapshot(1.0, 0.0, snap_test)
t_one = time.time() - start

t_total_h = t_one * 8760 * 72 / 3600
print(f"Temps par optimisation : {t_one:.3f} s")
print(f"Temps total estimé     : {t_total_h:.1f} heures")

In [ ]:
# FFOR soutenu en hiver a 14 h - sans BelalpSolar
# La loi lineaire d'autonomie HP definie plus haut est conservee.
jour_hiver = "2023-01-15"
heure_depart_hiver = 14
start_ts_hiver = pd.Timestamp(f"{jour_hiver} {heure_depart_hiver:02d}:00")


def build_winter_snapshot(scenario_selected, duration_h):
    idx = indices_pour_duree(start_ts_hiver, duration_h)
    scenario_nodes = list(map(int, scenario_selected["nodes"]))
    p_hp_min = (
        P_hp_dt_df_soutenu.iloc[idx]
        .max()
        .reindex(scenario_nodes, fill_value=0.0)
        .to_dict()
    )
    hp_factor = hp_sustained_flex_factor(duration_h)
    if FFOR_LOAD_MODE == "temporal":
        alpha_window, _ = load_alpha_for_indices(idx)
    else:
        alpha_window = 1.0
    return {
        "P_pv_available": build_pv_available_snapshot_for_scenario(
            scenario_selected, idx, method="min"
        ),
        "Q_pv_max": {
            bus: float(scenario_selected["Q_pv_max"].get(bus, 0.0))
            for bus in scenario_nodes
        },
        "P_hp_max": p_hp_min,
        "P_hp_upper": {
            bus: float(value) * (1.0 - hp_factor)
            for bus, value in p_hp_min.items()
        },
        "hp_flex_factor": hp_factor,
        "P_load": {
            bus: float(scenario_selected["P_load"].get(bus, 0.0))
            * alpha_window
            for bus in scenario_nodes
        },
        "Q_load": {
            bus: float(scenario_selected["Q_load"].get(bus, 0.0))
            * alpha_window
            for bus in scenario_nodes
        },
        "alpha": alpha_window,
        "time_indices": idx,
    }


def compute_winter_case(
    scenario_selected,
    solve_direction,
    validate_solution,
    output_prefix,
):
    case_results = {}
    summary_rows = []
    previous_polygon = None
    for duration_h, duration_label in durees_soutenues:
        snap = build_winter_snapshot(scenario_selected, duration_h)
        operating_point = compute_snapshot_operating_point(
            scenario_selected, snap
        )
        p_points, q_points = [], []
        for phi in angles_soutenu:
            solution = solve_direction(np.cos(phi), np.sin(phi), snap)
            ac_result = (
                validate_solution(solution)
                if solution is not None
                else {"valid": False}
            )
            if ac_result["valid"]:
                p_points.append(ac_result["P"])
                q_points.append(ac_result["Q"])
        if len(p_points) < 3:
            raise RuntimeError(f"FFOR hiver incomplet: {duration_label}")

        raw_polygon = convex_polygon(np.column_stack([
            [operating_point["P"] - value for value in p_points],
            [operating_point["Q"] - value for value in q_points],
        ]))
        polygon = (
            raw_polygon
            if previous_polygon is None
            else intersect_convex_polygons(previous_polygon, raw_polygon)
        )
        previous_polygon = polygon
        p_flex = np.append(polygon[:, 0], polygon[0, 0]).tolist()
        q_flex = np.append(polygon[:, 1], polygon[0, 1]).tolist()
        case_results[duration_h] = {
            "label": duration_label,
            "P_flex": p_flex,
            "Q_flex": q_flex,
        }
        summary_rows.append({
            "duration_h": duration_h,
            "duration_label": duration_label,
            "P_flex_pos_max_MW": max(p_flex),
            "P_flex_neg_min_MW": min(p_flex),
            "P_hp_reference_MW": sum(snap["P_hp_max"].values()),
            "hp_flex_factor": snap["hp_flex_factor"],
            "n_time_steps": len(snap["time_indices"]),
        })
        file_label = duration_label.replace(" ", "").replace(".", "")
        pd.DataFrame({"P_flex_MW": p_flex}).to_csv(
            output_dir / f"{output_prefix}_{start_ts_hiver:%Y%m%d_%Hh}_{file_label}.csv",
            index=False,
        )

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(
        output_dir / f"{output_prefix}_summary_{start_ts_hiver:%Y%m%d_%Hh}.csv",
        index=False,
    )
    return case_results, summary_df


def plot_winter_case(case_results, title, filename, colors):
    winter_tick_durations = [0.25, 1.0, 2.0, 4.0, 8.0]
    winter_tick_labels = ["15 min", "1 h", "2 h", "4 h", "8 h"]
    fig, ax_active = plt.subplots(figsize=(8.5, 5.5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    durations, p_pos, p_neg = [], [], []
    for duration_h, _ in durees_soutenues:
        data = case_results[duration_h]
        durations.append(duration_h)
        p_pos.append(max(data["P_flex"]))
        p_neg.append(min(data["P_flex"]))
    ax_active.fill_between(
        durations, 0, p_pos, color=colors[2], alpha=0.65,
        label="Positive flexibility",
    )
    ax_active.fill_between(
        durations, p_neg, 0, color=colors[0], alpha=0.75,
        label="Negative flexibility",
    )
    ax_active.plot(durations, p_pos, color=colors[-1], linewidth=2)
    ax_active.plot(durations, p_neg, color=colors[-2], linewidth=2)
    ax_active.axhline(0, color="black", linewidth=0.8)
    ax_active.set_xlabel("Sustained Duration [h]")
    ax_active.set_ylabel("Active Power Flexibility [MW]")
    ax_active.set_xticks(winter_tick_durations)
    ax_active.set_xticklabels(winter_tick_labels)
    ax_active.grid(True, alpha=0.35, linestyle="--")
    ax_active.legend()
    plt.tight_layout()
    plt.savefig(output_dir / filename, dpi=150, bbox_inches="tight")
    plt.show()


results_soutenus_hiver, summary_soutenu_hiver_df = compute_winter_case(
    scenario,
    solve_ffor_snapshot,
    validate_ac_solution,
    "FFOR_sustained_winter_without_belalp",
)
plot_winter_case(
    results_soutenus_hiver,
    f"Winter active power flexibility without BelalpSolar - {start_ts_hiver:%d.%m.%Y %H:%M}",
    f"FFOR_sustained_winter_without_belalp_{start_ts_hiver:%Y%m%d_%Hh}.png",
    ["#DBEAFE", "#BFDBFE", "#93C5FD", "#60A5FA", "#2563EB", "#0B3B75"],
)
display(summary_soutenu_hiver_df)


In [ ]:
# FFOR soutenu en hiver a 14 h - avec BelalpSolar
results_soutenus_hiver_belalp, summary_soutenu_hiver_belalp_df = compute_winter_case(
    scenario_soutenu_belalp,
    solve_ffor_snapshot_belalp,
    validate_ac_solution_belalp,
    "FFOR_sustained_winter_with_belalp",
)
plot_winter_case(
    results_soutenus_hiver_belalp,
    f"Winter active power flexibility with BelalpSolar - {start_ts_hiver:%d.%m.%Y %H:%M}",
    f"FFOR_sustained_winter_with_belalp_{start_ts_hiver:%Y%m%d_%Hh}.png",
    ["#DCFCE7", "#BBF7D0", "#86EFAC", "#4ADE80", "#16A34A", "#14532D"],
)
display(summary_soutenu_hiver_belalp_df)
